# 🚀 Zenith Codex Training
## Fine-tuning DeepSeek-Coder-6.7B на русский язык

Этот notebook обучит модель DeepSeek-Coder для Zenith.

**Требования:**
- Google Colab с GPU (T4 бесплатно)
- HuggingFace аккаунт (для сохранения модели)

**Время обучения:** ~2-4 часа на T4

## 1️⃣ Установка библиотек

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

## 2️⃣ Проверка GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if torch.cuda.get_device_properties(0).total_memory < 15e9:
    print("⚠️ Мало памяти! Перейди в Runtime -> Change runtime type -> T4 GPU")
else:
    print("✅ GPU готов!")

## 3️⃣ Логин в HuggingFace
Получи токен тут: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# Вставь свой токен HuggingFace (с правами write)
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxx"  # <-- ЗАМЕНИ НА СВОЙ ТОКЕН

login(token=HF_TOKEN)
print("✅ Залогинился в HuggingFace!")

## 4️⃣ Загрузка модели DeepSeek-Coder

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "deepseek-ai/deepseek-coder-6.7b-instruct"

# QLoRA config - экономит память
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("📥 Загружаю модель (это займёт 5-10 минут)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("✅ Модель загружена!")

## 5️⃣ Настройка LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6️⃣ Подготовка датасета
Используем русские инструкции + код

In [ ]:
from datasets import load_dataset, concatenate_datasets

print("📥 Загружаю датасеты...")

# Русские инструкции
ru_dataset = load_dataset("IlyaGusev/ru_turbo_alpaca", split="train[:5000]")

# Код датасет
code_dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:5000]")

print(f"✅ Русский датасет: {len(ru_dataset)} примеров")
print(f"✅ Код датасет: {len(code_dataset)} примеров")

In [ ]:
# Форматирование в стиле Zenith
def format_zenith_prompt(example):
    # Для русского датасета
    if 'instruction' in example and 'output' in example:
        instruction = example.get('instruction', '')
        inp = example.get('input', '')
        output = example.get('output', '')
        
        if inp:
            prompt = f"""### Инструкция:
{instruction}

### Ввод:
{inp}

### Ответ Zenith Codex:
{output}"""
        else:
            prompt = f"""### Инструкция:
{instruction}

### Ответ Zenith Codex:
{output}"""
    else:
        prompt = str(example)
    
    return {"text": prompt}

# Применяем форматирование
ru_formatted = ru_dataset.map(format_zenith_prompt, remove_columns=ru_dataset.column_names)
code_formatted = code_dataset.map(format_zenith_prompt, remove_columns=code_dataset.column_names)

# Объединяем
combined_dataset = concatenate_datasets([ru_formatted, code_formatted])
combined_dataset = combined_dataset.shuffle(seed=42)

print(f"✅ Итого: {len(combined_dataset)} примеров для обучения")
print("\n📝 Пример:")
print(combined_dataset[0]['text'][:500])

## 7️⃣ Обучение модели
⏱️ Это займёт 2-4 часа на T4

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Твой username на HuggingFace
HF_USERNAME = "your-username"  # <-- ЗАМЕНИ НА СВОЙ USERNAME
OUTPUT_MODEL = f"{HF_USERNAME}/zenith-codex-6.7b"

training_args = TrainingArguments(
    output_dir="./zenith-codex",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    push_to_hub=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=combined_dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
)

print("🚀 Начинаю обучение...")
print("⏱️ Это займёт 2-4 часа. Можешь оставить вкладку открытой.")

trainer.train()

## 8️⃣ Сохранение модели

In [ ]:
print("💾 Сохраняю модель...")

# Сохраняем LoRA адаптер
model.save_pretrained("./zenith-codex-lora")
tokenizer.save_pretrained("./zenith-codex-lora")

# Загружаем на HuggingFace
model.push_to_hub(OUTPUT_MODEL, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_MODEL, token=HF_TOKEN)

print(f"✅ Модель сохранена: https://huggingface.co/{OUTPUT_MODEL}")

## 9️⃣ Тестирование модели

In [ ]:
# Тест модели
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые промпты
test_prompts = [
    "### Инструкция:\nНапиши функцию на Python для сортировки списка\n\n### Ответ Zenith Codex:",
    "### Инструкция:\nОбъясни что такое рекурсия простыми словами\n\n### Ответ Zenith Codex:",
    "### Инструкция:\nНапиши HTML страницу с кнопкой\n\n### Ответ Zenith Codex:",
]

print("🧪 Тестирование модели:\n")
for prompt in test_prompts:
    print("="*50)
    response = generate_response(prompt)
    print(response)
    print()

## ✅ Готово!

Твоя модель **Zenith Codex** обучена и загружена на HuggingFace!

Теперь можешь:
1. Использовать её через HuggingFace Inference API
2. Скачать и запустить локально
3. Подключить к Zenith Sync